# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [3]:
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026


### Creating the model again from ML-09

In [4]:
# recreate the ML-08 dataset
model_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ) > 0
            THEN
                CAST(
                    SUM(gsc_sum_position) FILTER (
                        WHERE gsc_data_available IS TRUE
                    ) AS DOUBLE
                )
                /
                SUM(gsc_impressions) FILTER (
                    WHERE gsc_data_available IS TRUE
                )
            ELSE NULL
        END AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_mar,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_days_mar

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.avg_position_feb,
    m.clicks_mar,
    m.available_days_mar,

    CASE
        WHEN m.available_days_mar > 0
             AND COALESCE(m.clicks_mar, 0) = 0
        THEN 1
        ELSE 0
    END AS march_zero_click

FROM feb f

JOIN '{DIM_CONTENT}' c
    ON f.content_hash_id = c.content_hash_id

JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3
    AND f.avg_position_feb IS NOT NULL
    AND c.is_published IS TRUE
""").df()

print("Rows:", len(model_df))
display(model_df.head())

Rows: 29362


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,clicks_mar,available_days_mar,march_zero_click
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4270.0,7.0,5.960187,7.0,31,0
1,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5271.0,4.0,7.125783,6.0,31,0
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,6690.0,19.0,7.226756,13.0,31,0
3,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,4314.0,14.0,6.424200,20.0,31,0
4,client_73cda7b4e4f265ea,content_20403327d8d9374c,2756.0,8.0,6.726415,10.0,31,0


In [5]:
# Before — random row split
FEATURES = [
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = order[:k]
    return np.mean(np.asarray(y_true)[top_k])


train_random, test_random = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df["march_zero_click"]
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),
])

random_model.fit(
    train_random[FEATURES],
    train_random["march_zero_click"]
)

random_scores = random_model.predict_proba(
    test_random[FEATURES]
)[:, 1]

random_p20 = precision_at_k(
    test_random["march_zero_click"],
    random_scores,
    20
)

random_p50 = precision_at_k(
    test_random["march_zero_click"],
    random_scores,
    50
)

print(f"Random row split Precision@20: {random_p20:.3f}")
print(f"Random row split Precision@50: {random_p50:.3f}")

Random row split Precision@20: 0.200
Random row split Precision@50: 0.200


In [6]:
# After — client-grouped split
clients = model_df["client_hash_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_grouped = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

test_grouped = model_df[
    model_df["client_hash_id"].isin(test_clients)
].copy()

grouped_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),
])

grouped_model.fit(
    train_grouped[FEATURES],
    train_grouped["march_zero_click"]
)

grouped_scores = grouped_model.predict_proba(
    test_grouped[FEATURES]
)[:, 1]

grouped_p20 = precision_at_k(
    test_grouped["march_zero_click"],
    grouped_scores,
    20
)

grouped_p50 = precision_at_k(
    test_grouped["march_zero_click"],
    grouped_scores,
    50
)

print(f"Grouped split Precision@20: {grouped_p20:.3f}")
print(f"Grouped split Precision@50: {grouped_p50:.3f}")

print(
    "Client overlap:",
    len(
        set(train_grouped["client_hash_id"])
        &
        set(test_grouped["client_hash_id"])
    )
)

Grouped split Precision@20: 0.300
Grouped split Precision@50: 0.340
Client overlap: 0


## 1. Ranked actions + reason codes

The playbook turns the validated zero-click risk ranking into a human-reviewed content queue.

The primary action is **REVIEW_ZERO_CLICK_RISK**. It identifies pages that receive high model scores and therefore deserve earlier review by a content/SEO practitioner.

Reason codes are based on the available February signals:

- `HIGH_ZERO_CLICK_RISK` — high model score places the page near the top of the review queue.
- `LOW_FEB_CLICKS` — the page had relatively few clicks during February.
- `LOW_FEB_IMPRESSIONS` — the page had relatively low search visibility during February.
- `WEAKER_POSITION` — the February average position was relatively weak.

The score is a prioritization signal, not a claim that the page will definitely receive zero clicks. The final content action is decided by a human reviewer.

In [7]:
# Reuse the validated ML-09 model outputs.
# These variables should already exist if the notebook is run after the ML-09 setup:
# test_grouped, grouped_model, grouped_scores, FEATURES

playbook = test_grouped[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "avg_position_feb",
        "march_zero_click",
    ]
].copy()

playbook["model_score"] = grouped_scores

print("Playbook rows:", len(playbook))
display(playbook.head())

Playbook rows: 2522


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,march_zero_click,model_score
3596,client_400c21c81c8b46ef,content_5a6ffbb6e1907e6b,926.0,6.0,6.771058,0,0.066907
3597,client_400c21c81c8b46ef,content_40df6145d59c71c5,122.0,4.0,6.991803,1,0.109505
3598,client_400c21c81c8b46ef,content_f380474d30b612b2,873.0,4.0,7.003436,0,0.090700
3599,client_400c21c81c8b46ef,content_907ab15f08b494bf,2632.0,7.0,4.343085,0,0.034872
3600,client_400c21c81c8b46ef,content_edadadb84b728d61,456.0,3.0,7.964912,1,0.117926


In [8]:
# Reason-code code cell
# Percentile thresholds are calculated only within the evaluated queue.
# They are used for prioritization, not as causal thresholds.

click_threshold = playbook["clicks_feb"].quantile(0.25)
impression_threshold = playbook["impressions_feb"].quantile(0.25)
position_threshold = playbook["avg_position_feb"].quantile(0.75)

def reason_code(row):
    if row["model_score"] >= playbook["model_score"].quantile(0.90):
        return "HIGH_ZERO_CLICK_RISK"
    elif row["clicks_feb"] <= click_threshold:
        return "LOW_FEB_CLICKS"
    elif row["impressions_feb"] <= impression_threshold:
        return "LOW_FEB_IMPRESSIONS"
    elif row["avg_position_feb"] >= position_threshold:
        return "WEAKER_POSITION"
    else:
        return "MODEL_PRIORITY"

playbook["reason_code"] = playbook.apply(reason_code, axis=1)

playbook["action"] = "REVIEW_ZERO_CLICK_RISK"

playbook = playbook.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

playbook["rank"] = np.arange(1, len(playbook) + 1)

display(
    playbook[
        [
            "rank",
            "content_hash_id",
            "model_score",
            "reason_code",
            "action",
        ]
    ].head(20)
)

,rank,content_hash_id,model_score,reason_code,action
0,1,content_e503a367a27d1940,0.218524,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
1,2,content_8615b34504393ec6,0.205031,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
2,3,content_3345847e056746fc,0.203992,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
3,4,content_de8bd70a5a760ecd,0.197235,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
4,5,content_7b96894b2550c273,0.183915,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
5,6,content_6cad17b1ec544c83,0.183337,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
6,7,content_1339ce1653f96e48,0.180651,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
7,8,content_07c70f0084fa59ff,0.180102,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
8,9,content_3af1e8c7cda1297d,0.179935,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK
9,10,content_e7fa2f7aa5ef29f7,0.179162,HIGH_ZERO_CLICK_RISK,REVIEW_ZERO_CLICK_RISK


## 2. Intended use and limits

### Intended use

The playbook is intended to help an SEO or content practitioner prioritize pages for review. A high-ranked page receives earlier attention because its February search signals produced a higher measured model score.

The output is decision-support: it helps order the review queue but does not decide what content change should be made.

### Archetype → action mapping

| Observed situation | Suggested human action |
|---|---|
| High model score | Review page before lower-ranked pages |
| Low February clicks | Inspect search demand, snippets, intent alignment, and page usefulness |
| Low February impressions | Check search visibility, indexing, targeting, and query coverage |
| Weak average position | Review relevance, content coverage, internal linking, and SERP competition |
| Multiple weak signals | Prioritize a deeper content/SEO review |

### Limits

The model uses only three February search-performance features. It does not directly observe content quality, search intent, SERP features, seasonality, competitor changes, technical SEO issues, or the reason a page receives zero clicks.

The observed relationship is therefore directional. A high score should trigger review, not automatic editing, deletion, rewriting, or publication.

The model was evaluated on this dataset and validation design. Its measured performance does not establish the same performance for every future client or future period.

## 3. Human review + the no-go list

### Human review rules

Before taking action on a ranked page, a reviewer should:

1. Confirm that the page is currently published and still relevant.
2. Inspect the page and its search intent manually.
3. Check whether the February signals represent a meaningful period for the page.
4. Look for technical or indexing issues before changing the content.
5. Check whether seasonality or a temporary event could explain the observed search behavior.
6. Decide the actual content action only after this review.

### No-go list

The model should NOT automatically:

- rewrite or publish content;
- delete a page;
- redirect a URL;
- change canonical tags;
- change metadata without review;
- change internal links automatically;
- declare a page "bad" or "low quality";
- infer a causal reason for zero clicks;
- make client-facing recommendations without human review.

The model produces a ranked review queue, not an autonomous content-management system.

In [9]:
# Final playbook feature audit

forbidden = {
    "march_zero_click",
    "clicks_mar",
    "available_days_mar",
    "trend_direction",
    "march_zero_click_rate",
    "future_clicks",
    "future_impressions",
}

playbook_features = FEATURES

leaked = [
    col for col in playbook_features
    if col in forbidden
]

print("Decision-time features:")
for col in playbook_features:
    print("-", col)

print("\nForbidden features found:", leaked)

assert not leaked, f"Leakage detected: {leaked}"

print("\nPlaybook feature audit: PASS")

Decision-time features:
- impressions_feb
- clicks_feb
- avg_position_feb

Forbidden features found: []

Playbook feature audit: PASS


## 4. Monitoring / retrain triggers

The recommendations should be reviewed periodically rather than assumed to remain valid indefinitely.

### Monitoring

Monitor:

- Precision@20 and Precision@50 on newly available labeled months.
- The proportion of pages receiving very high model scores.
- The distribution of February impressions, clicks, and average position.
- The proportion of rows with missing or unavailable search data.
- Whether the relationship between February signals and the following month's zero-click outcome changes.

### Retrain / review triggers

A model review should be considered if:

- Precision@20 or Precision@50 shows a sustained decline across newly evaluated months.
- Feature distributions shift materially from the training data.
- Search-data availability changes substantially.
- The meaning or definition of the zero-click outcome changes.
- The content population changes enough that the existing validation population is no longer representative.

These are monitoring triggers, not automatic retraining commands. A human should inspect the cause of the change before retraining.

In [10]:
monitoring_receipt = pd.DataFrame([
    {
        "metric": "Precision@20",
        "value": grouped_p20,
        "current_reference": "ML-09 client-grouped validation"
    },
    {
        "metric": "Precision@50",
        "value": grouped_p50,
        "current_reference": "ML-09 client-grouped validation"
    },
    {
        "metric": "rows_in_review_queue",
        "value": len(playbook),
        "current_reference": "Current evaluated population"
    },
])

display(monitoring_receipt)

,metric,value,current_reference
0,Precision@20,0.30,ML-09 client-grouped validation
1,Precision@50,0.34,ML-09 client-grouped validation
2,rows_in_review_queue,2522.00,Current evaluated population


## 5. Exports for the paper

The ranked queue is exported so that the same measured output can be reused in the recommendations section of the research paper.

The CSV is generated by the notebook and should not contain client names, URLs, or private query information.

The exported queue contains the ranking, model score, reason code, action, and decision-time signals needed to reproduce the prioritization logic.

In [11]:
from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_PATH = OUTPUT_DIR / "w07_action_playbook_queue.csv"

export_columns = [
    "rank",
    "content_hash_id",
    "model_score",
    "reason_code",
    "action",
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
]

playbook[export_columns].to_csv(
    QUEUE_PATH,
    index=False
)

print(f"Exported queue: {QUEUE_PATH}")
print(f"Rows exported: {len(playbook)}")

assert QUEUE_PATH.exists()

Exported queue: work/outputs/w07_action_playbook_queue.csv
Rows exported: 2522


In [12]:
# paper-ready summary JSON
import json

metrics = {
    "feature_window": "2026-02",
    "outcome_window": "2026-03",
    "model": "Logistic Regression",
    "validation": "client-grouped",
    "features": FEATURES,
    "precision_at_20": float(grouped_p20),
    "precision_at_50": float(grouped_p50),
    "queue_action": "REVIEW_ZERO_CLICK_RISK",
    "leakage_check": "PASS",
}

METRICS_PATH = OUTPUT_DIR / "w07_action_playbook_metrics.json"

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics receipt: {METRICS_PATH}")

Saved metrics receipt: work/outputs/w07_action_playbook_metrics.json


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] The ranked queue contains a reason code and human-review action.
- [x] The action is decision-support rather than automatic content modification.
- [x] The archetype → action mapping is documented.
- [x] Intended use and limitations are stated.
- [x] Human review rules are documented.
- [x] No-go cases are explicitly listed.
- [x] Monitoring and retrain triggers are defined.
- [x] The queue is exported to `work/outputs/`.
- [x] The exported queue contains only decision-time fields and model outputs.
- [x] No future-window or label-derived feature is used for ranking.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [x] The notebook runs successfully with Runtime → Run all.
- [x] Committed to `work/notebooks/w07_action_playbook.ipynb`.